# LeetCode 892 — Surface Area of 3D Shapes bằng Python

**Link đề:** https://leetcode.com/problems/surface-area-of-3d-shapes/
**Độ khó:** Easy · **Chủ đề:** Array, Math, Geometry, Matrix

```python
class Solution:
    def surfaceArea(self, grid: list[list[int]]) -> int:
        ...
```

## Mục tiêu học
1. Tính diện tích bề mặt bằng tư duy **"mặt lộ ra = tổng mặt − mặt bị dán"**.
2. Duyệt ma trận với **4 hướng kề** (`dirs`), kiểm tra **biên** để không tràn chỉ số.
3. Đi từ mô phỏng từng khối lập phương (set tọa độ 3D) → công thức theo tháp `4v + 2` → mẹo chỉ xét hàng xóm **trên/trái** và trừ `2·min`.
4. Trace từng ô, chứng minh không đếm trùng, phân tích `O(n²)`.
5. Phân biệt với bài **883 Projection Area** (bóng chiếu ≠ bề mặt), luyện 6 bài tập.

## 1. Đề bài và ví dụ

Lưới `n × n`, ô `(i, j)` có tháp `v = grid[i][j]` khối lập phương `1×1×1`. Các khối **kề nhau trực tiếp** được **dán** lại, tạo thành các khối hình bất kỳ.

Trả về **tổng diện tích bề mặt** của các khối. **Mặt đáy** cũng được tính.

| grid | Output |
|---|---:|
| `[[1,2],[3,4]]` | 34 |
| `[[1,1,1],[1,0,1],[1,1,1]]` | 32 |
| `[[2,2,2],[2,1,2],[2,2,2]]` | 46 |

**Constraints:** `n == grid.length == grid[i].length`, `1 <= n <= 50`, `0 <= grid[i][j] <= 50`.

Lưu ý ví dụ 2: ô giữa trống (0) → các mặt **bên trong** của "cái giếng" cũng là bề mặt.

### 1.1. Tính tay ví dụ nhỏ nhất: một khối, rồi hai khối dán nhau

- 1 khối lập phương: **6** mặt.
- 2 khối đặt cạnh nhau: `6 + 6 = 12`, nhưng **1 cặp mặt** áp vào nhau bị che → mỗi khối mất 1 mặt → `12 − 2 = 10`.
- Tháp cao `v` (chồng `v` khối): `6v` trừ `2(v − 1)` mặt dán theo chiều đứng → `4v + 2` (4 mặt bên mỗi tầng + đỉnh + đáy).

Tháp `v = 2`: `4·2 + 2 = 10` ✔ (trùng với 2 khối dán nhau).

## 2. Nền tảng Python cần biết

### 2.1. Duyệt ma trận và 4 hướng kề

Hàng xóm của ô `(i, j)` theo 4 hướng: lên `(i-1, j)`, xuống `(i+1, j)`, trái `(i, j-1)`, phải `(i, j+1)`. Ta lưu các độ lệch vào list `dirs` để viết vòng lặp gọn.

In [1]:
dirs = [(-1, 0), (1, 0), (0, -1), (0, 1)]    # lên, xuống, trái, phải
i, j = 1, 1
for di, dj in dirs:
    print(f"({i},{j}) + ({di:+d},{dj:+d}) -> ({i + di},{j + dj})")

(1,1) + (-1,+0) -> (0,1)
(1,1) + (+1,+0) -> (2,1)
(1,1) + (+0,-1) -> (1,0)
(1,1) + (+0,+1) -> (1,2)


### 2.2. Kiểm tra biên: `0 <= ni < n and 0 <= nj < n`

Ô ở mép có hàng xóm **ngoài lưới**. Trong Python, chỉ số âm **không lỗi** mà lấy từ cuối (`grid[-1]` là hàng cuối) — rất nguy hiểm vì cho kết quả sai âm thầm! Nên luôn kiểm tra biên.

Python cho phép viết so sánh chuỗi `0 <= ni < n` (tương đương `0 <= ni and ni < n`).

In [2]:
grid = [[1, 2], [3, 4]]
n = len(grid)
print("grid[-1][0] =", grid[-1][0], "<- KHÔNG lỗi, lấy hàng cuối! (sai âm thầm)")
for ni, nj in [(-1, 0), (0, 1), (2, 0)]:
    inside = 0 <= ni < n and 0 <= nj < n
    print(f"({ni},{nj}) trong lưới? {inside}")

grid[-1][0] = 3 <- KHÔNG lỗi, lấy hàng cuối! (sai âm thầm)
(-1,0) trong lưới? False
(0,1) trong lưới? True
(2,0) trong lưới? False


### 2.3. `min(a, b)` và `max(x, 0)`

- Hai tháp kề nhau cao `a` và `b`: phần **áp vào nhau** cao `min(a, b)` → bị che `min(a, b)` mặt **mỗi bên**.
- Phần tháp `a` **lộ ra** về phía tháp `b`: `max(a − b, 0)` (không âm).

In [3]:
for a, c in [(3, 1), (1, 3), (2, 2), (4, 0)]:
    print(f"a={a}, b={c}: áp nhau min={min(a, c)}, a lộ ra phía b = max(a-b,0)={max(a - c, 0)}")

a=3, b=1: áp nhau min=1, a lộ ra phía b = max(a-b,0)=2
a=1, b=3: áp nhau min=1, a lộ ra phía b = max(a-b,0)=0
a=2, b=2: áp nhau min=2, a lộ ra phía b = max(a-b,0)=0
a=4, b=0: áp nhau min=0, a lộ ra phía b = max(a-b,0)=4


### 2.4. `set` các tọa độ 3D (dùng cho baseline)

Tuple `(x, y, z)` có thể làm phần tử `set`. Kiểm tra `(x, y, z) in cubes` là `O(1)` trung bình.

In [4]:
cubes = {(0, 0, 0), (0, 0, 1)}
print((0, 0, 1) in cubes, (5, 5, 5) in cubes)

True False


## 3. Cách 1 — Baseline: mô phỏng từng khối lập phương

Đặt mọi khối vào `set` tọa độ `(i, j, z)` với `z = 0..v-1`. Với **mỗi khối**, xét 6 hướng (±x, ±y, ±z): nếu ô bên cạnh **không có khối** → mặt đó lộ ra → `+1`.

Không cần nghĩ công thức; đúng theo định nghĩa "bề mặt".

In [5]:
def surface_area_simulate(grid: list[list[int]]) -> int:
    n = len(grid)
    cubes = {(i, j, z) for i in range(n) for j in range(n) for z in range(grid[i][j])}
    dirs6 = [(1, 0, 0), (-1, 0, 0), (0, 1, 0), (0, -1, 0), (0, 0, 1), (0, 0, -1)]
    area = 0
    for (x, y, z) in cubes:
        for dx, dy, dz in dirs6:
            if (x + dx, y + dy, z + dz) not in cubes:
                area += 1
    return area

for g in ([[1, 2], [3, 4]], [[1, 1, 1], [1, 0, 1], [1, 1, 1]], [[2, 2, 2], [2, 1, 2], [2, 2, 2]]):
    print(g, "->", surface_area_simulate(g))

[[1, 2], [3, 4]] -> 34
[[1, 1, 1], [1, 0, 1], [1, 1, 1]] -> 32
[[2, 2, 2], [2, 1, 2], [2, 2, 2]] -> 46


Mặt đáy tự động được tính: khối ở `z = 0` có hàng xóm `z = -1` không bao giờ nằm trong set.

### 3.1. Nút thắt
Số khối tối đa `50·50·50 = 125 000`, mỗi khối 6 lần tra set → ~750 000 thao tác: vẫn chạy được, nhưng tốn bộ nhớ `O(n²·H)` và phụ thuộc chiều cao `H`. Nếu `v` lên tới `10^9` thì không thể. Ta cần làm việc ở mức **tháp**, không phải mức **khối**.

## 4. Insight: tính theo tháp, trừ phần bị che

**Bước 1 — một tháp đứng một mình:** `4v + 2` (nếu `v > 0`), `0` nếu `v = 0`.

**Bước 2 — hai tháp kề nhau** cao `a`, `b`: dọc theo mặt tiếp xúc, có `min(a, b)` cặp mặt áp vào nhau. Mỗi cặp làm **mất 2 mặt** (1 của mỗi tháp).

```text
Diện tích = Σ (4v + 2 với v > 0)  −  Σ_{cặp kề nhau} 2·min(a, b)
```

Kiểm tra ví dụ 1 `[[1,2],[3,4]]`:
- Tổng tháp riêng: `(4+2) + (8+2) + (12+2) + (16+2) = 6 + 10 + 14 + 18 = 48`.
- Cặp kề: (1,2) ngang → min 1; (3,4) ngang → 3; (1,3) dọc → 1; (2,4) dọc → 2. Tổng min = 7 → trừ `2·7 = 14`.
- `48 − 14 = 34` ✔.

In [6]:
towers = [4 * v + 2 for row in [[1, 2], [3, 4]] for v in row if v > 0]
print("Mỗi tháp riêng:", towers, "-> tổng", sum(towers))
pairs = [(1, 2), (3, 4), (1, 3), (2, 4)]
print("Trừ:", [2 * min(a, c) for a, c in pairs], "-> tổng", sum(2 * min(a, c) for a, c in pairs))
print("Kết quả:", sum(towers) - sum(2 * min(a, c) for a, c in pairs))

Mỗi tháp riêng: [6, 10, 14, 18] -> tổng 48
Trừ: [2, 6, 2, 4] -> tổng 14
Kết quả: 34


## 5. Cách 2 — Mỗi tháp xét đủ 4 hàng xóm (đếm phần lộ ra)

Với mỗi tháp `v > 0`:
- `+2` (đỉnh + đáy).
- Với mỗi hướng: hàng xóm cao `nv` (hoặc `0` nếu ra ngoài lưới) → mặt bên lộ ra `max(v − nv, 0)`.

Ý nghĩa: phần tháp **cao hơn** hàng xóm thì lộ ra; phần thấp hơn hoặc bằng bị hàng xóm che.

In [7]:
def surface_area_four_dirs(grid: list[list[int]]) -> int:
    n = len(grid)
    area = 0
    for i in range(n):
        for j in range(n):
            v = grid[i][j]
            if v == 0:
                continue
            area += 2                                    # đỉnh + đáy
            for di, dj in ((-1, 0), (1, 0), (0, -1), (0, 1)):
                ni, nj = i + di, j + dj
                nv = grid[ni][nj] if 0 <= ni < n and 0 <= nj < n else 0
                area += max(v - nv, 0)                   # mặt bên lộ ra hướng này
    return area

for g in ([[1, 2], [3, 4]], [[1, 1, 1], [1, 0, 1], [1, 1, 1]], [[2, 2, 2], [2, 1, 2], [2, 2, 2]]):
    print(g, "->", surface_area_four_dirs(g))

[[1, 2], [3, 4]] -> 34
[[1, 1, 1], [1, 0, 1], [1, 1, 1]] -> 32
[[2, 2, 2], [2, 1, 2], [2, 2, 2]] -> 46


`continue` bỏ qua phần còn lại của vòng hiện tại, sang ô tiếp theo. Ô trống không có đỉnh/đáy và không có mặt bên của chính nó (mặt bên của hàng xóm hướng vào ô trống do hàng xóm tự đếm).

## 6. Cách 3 — Lời giải chính: `4v + 2` rồi chỉ trừ với hàng xóm **trên** và **trái**

Mỗi cặp kề nhau chỉ cần trừ **một lần**. Nếu tại ô `(i, j)` ta chỉ nhìn ô **trên** `(i-1, j)` và ô **trái** `(i, j-1)`, thì mỗi cặp dọc được xử lý đúng một lần (bởi ô ở dưới), mỗi cặp ngang đúng một lần (bởi ô bên phải).

Lợi ích: không cần kiểm tra biên phải/dưới, chỉ cần `i > 0` và `j > 0`.

In [8]:
def surface_area_up_left(grid: list[list[int]]) -> int:
    n = len(grid)
    area = 0
    for i in range(n):
        for j in range(n):
            v = grid[i][j]
            if v:
                area += 4 * v + 2
            if i > 0:
                area -= 2 * min(v, grid[i - 1][j])     # dán với ô trên
            if j > 0:
                area -= 2 * min(v, grid[i][j - 1])     # dán với ô trái
    return area

for g in ([[1, 2], [3, 4]], [[1, 1, 1], [1, 0, 1], [1, 1, 1]], [[2, 2, 2], [2, 1, 2], [2, 2, 2]]):
    print(g, "->", surface_area_up_left(g))

[[1, 2], [3, 4]] -> 34
[[1, 1, 1], [1, 0, 1], [1, 1, 1]] -> 32
[[2, 2, 2], [2, 1, 2], [2, 2, 2]] -> 46


Chú ý: phần trừ đặt **ngoài** `if v:`. Khi `v = 0` thì `min(0, ...) = 0` nên trừ 0 — vô hại, viết vậy cho gọn.

## 7. TRACE từng bước

### 7.1. Bảng trace cách 3 với `[[1,2],[3,4]]`

| (i,j) | v | +4v+2 | trên | −2·min(trên) | trái | −2·min(trái) | area sau |
|---|---:|---:|---:|---:|---:|---:|---:|
| (0,0) | 1 | +6 | — | 0 | — | 0 | 6 |
| (0,1) | 2 | +10 | — | 0 | 1 | −2 | 14 |
| (1,0) | 3 | +14 | 1 | −2 | — | 0 | 26 |
| (1,1) | 4 | +18 | 2 | −4 | 3 | −6 | **34** |

In [9]:
def trace_surface(grid):
    n = len(grid)
    area = 0
    for i in range(n):
        for j in range(n):
            v = grid[i][j]
            add = 4 * v + 2 if v else 0
            up = 2 * min(v, grid[i - 1][j]) if i > 0 else 0
            left = 2 * min(v, grid[i][j - 1]) if j > 0 else 0
            area += add - up - left
            up_s = grid[i - 1][j] if i > 0 else "-"
            left_s = grid[i][j - 1] if j > 0 else "-"
            print(f"({i},{j}) v={v}: +{add:<3} | trên={up_s} -> -{up:<2} | trái={left_s} -> -{left:<2} | area={area}")
    print("KẾT QUẢ:", area)
    return area

_ = trace_surface([[1, 2], [3, 4]])

(0,0) v=1: +6   | trên=- -> -0  | trái=- -> -0  | area=6
(0,1) v=2: +10  | trên=- -> -0  | trái=1 -> -2  | area=14
(1,0) v=3: +14  | trên=1 -> -2  | trái=- -> -0  | area=26
(1,1) v=4: +18  | trên=2 -> -4  | trái=3 -> -6  | area=34
KẾT QUẢ: 34


### 7.2. Trace ví dụ "cái giếng" có ô 0 ở giữa

Ô giữa `v = 0` không cộng gì, và cũng không trừ gì với hàng xóm (min = 0). Nhờ vậy các mặt bên hướng vào giếng của 4 ô xung quanh **vẫn được giữ** → kết quả 32.

In [10]:
_ = trace_surface([[1, 1, 1], [1, 0, 1], [1, 1, 1]])

(0,0) v=1: +6   | trên=- -> -0  | trái=- -> -0  | area=6
(0,1) v=1: +6   | trên=- -> -0  | trái=1 -> -2  | area=10
(0,2) v=1: +6   | trên=- -> -0  | trái=1 -> -2  | area=14
(1,0) v=1: +6   | trên=1 -> -2  | trái=- -> -0  | area=18
(1,1) v=0: +0   | trên=1 -> -0  | trái=1 -> -0  | area=18
(1,2) v=1: +6   | trên=1 -> -2  | trái=0 -> -0  | area=22
(2,0) v=1: +6   | trên=1 -> -2  | trái=- -> -0  | area=26
(2,1) v=1: +6   | trên=0 -> -0  | trái=1 -> -2  | area=30
(2,2) v=1: +6   | trên=1 -> -2  | trái=1 -> -2  | area=32
KẾT QUẢ: 32


### 7.3. Trace cách 2 cho một ô: nhìn rõ 4 hướng

Lấy ô `(1,1)` có `v = 1` trong `[[2,2,2],[2,1,2],[2,2,2]]`: mọi hàng xóm cao 2 ≥ 1 → không mặt bên nào lộ → ô này chỉ góp `2` (đỉnh + đáy).

In [11]:
def trace_cell_four_dirs(grid, i, j):
    n = len(grid)
    v = grid[i][j]
    total = 2 if v else 0
    print(f"Ô ({i},{j}) v={v}: đỉnh+đáy = {total}")
    for name, (di, dj) in zip(["trên", "dưới", "trái", "phải"], [(-1, 0), (1, 0), (0, -1), (0, 1)]):
        ni, nj = i + di, j + dj
        nv = grid[ni][nj] if 0 <= ni < n and 0 <= nj < n else 0
        exposed = max(v - nv, 0) if v else 0
        total += exposed
        print(f"  {name:5}: hàng xóm cao {nv} -> lộ {exposed}")
    print("  Tổng ô này góp:", total)

trace_cell_four_dirs([[2, 2, 2], [2, 1, 2], [2, 2, 2]], 1, 1)
trace_cell_four_dirs([[2, 2, 2], [2, 1, 2], [2, 2, 2]], 0, 0)

Ô (1,1) v=1: đỉnh+đáy = 2
  trên : hàng xóm cao 2 -> lộ 0
  dưới : hàng xóm cao 2 -> lộ 0
  trái : hàng xóm cao 2 -> lộ 0
  phải : hàng xóm cao 2 -> lộ 0
  Tổng ô này góp: 2
Ô (0,0) v=2: đỉnh+đáy = 2
  trên : hàng xóm cao 0 -> lộ 2
  dưới : hàng xóm cao 2 -> lộ 0
  trái : hàng xóm cao 0 -> lộ 2
  phải : hàng xóm cao 2 -> lộ 0
  Tổng ô này góp: 6


## 8. Code nộp LeetCode

In [12]:
class Solution:
    def surfaceArea(self, grid: list[list[int]]) -> int:
        n = len(grid)
        area = 0
        for i in range(n):
            for j in range(n):
                v = grid[i][j]
                if v:
                    area += 4 * v + 2
                if i > 0:
                    area -= 2 * min(v, grid[i - 1][j])
                if j > 0:
                    area -= 2 * min(v, grid[i][j - 1])
        return area

sol = Solution()
print(sol.surfaceArea([[1, 2], [3, 4]]), sol.surfaceArea([[1, 1, 1], [1, 0, 1], [1, 1, 1]]),
      sol.surfaceArea([[2, 2, 2], [2, 1, 2], [2, 2, 2]]))

34 32 46


### 8.1. Bản nộp 4 hướng (đếm phần lộ ra)

In [13]:
class SolutionFourDirs:
    def surfaceArea(self, grid: list[list[int]]) -> int:
        n = len(grid)
        area = 0
        for i in range(n):
            for j in range(n):
                v = grid[i][j]
                if not v:
                    continue
                area += 2
                for ni, nj in ((i - 1, j), (i + 1, j), (i, j - 1), (i, j + 1)):
                    nv = grid[ni][nj] if 0 <= ni < n and 0 <= nj < n else 0
                    area += max(v - nv, 0)
        return area

print(SolutionFourDirs().surfaceArea([[1, 2], [3, 4]]))

34


## 9. Vì sao đúng?

**Cách 3.** Bắt đầu từ tổng mặt của các tháp **đứng riêng** (mỗi tháp đã trừ đúng các mặt dán theo chiều đứng: `6v − 2(v−1) = 4v + 2`). Hai khối thuộc **hai tháp khác nhau** chỉ có thể dán nhau nếu hai tháp **kề ngang/dọc** và cùng tầng `z`; số tầng chung là `min(a, b)`; mỗi tầng chung che 2 mặt. Mỗi cặp tháp kề được xét **đúng một lần** (bởi ô dưới hoặc ô phải) → không đếm trùng, không bỏ sót.

**Invariant cách 3:** sau khi xử lý các ô theo thứ tự hàng → cột tới `(i, j)`, `area` = diện tích bề mặt của khối hình chỉ gồm các ô đã xử lý.

**Cách 2.** Mặt bên hướng về phía hàng xóm ở tầng `z` lộ ra ⇔ hàng xóm không có khối ở tầng `z` ⇔ `z ≥ nv`. Với `0 ≤ z < v`, số tầng như vậy là `max(v − nv, 0)`. Đỉnh và đáy của tháp luôn lộ (không có gì dán trên đỉnh hay dưới đáy) → `+2`.

**Hai cách bằng nhau:** với cặp kề `(a, b)`: cách 2 đếm `max(a−b,0) + max(b−a,0) = |a − b|`; cách 3 đếm `a + b − 2·min(a, b) = |a − b|` (phần 4v của mỗi tháp góp `a` và `b` cho hướng đó). Khớp! (Với `a = 0` hoặc `b = 0` cũng đúng.)

## 10. Độ phức tạp và so sánh

`n` = kích thước lưới, `H` = chiều cao tối đa.

| Cách | Ý tưởng | Thời gian | Bộ nhớ thêm | Ghi chú |
|---|---|---|---|---|
| 1. Mô phỏng khối | 6 hướng mỗi khối | `O(n²·H)` | `O(n²·H)` | Kiểm chứng tốt, phụ thuộc chiều cao |
| 2. 4 hướng mỗi tháp | `+2 + Σ max(v−nv,0)` | `O(n²)` | `O(1)` | Trực quan "phần lộ ra" |
| 3. Trên/trái | `4v+2 − 2·min` | `O(n²)` | `O(1)` | **Gọn nhất**, ít kiểm biên |

Mỗi ô xem tối đa 4 (cách 2) hoặc 2 (cách 3) hàng xóm → hằng số nhỏ.

## 11. Bộ test: mẫu đề, biên, đối chiếu

In [14]:
def check(name, actual, expected):
    assert actual == expected, (name, actual, expected)
    print(f"PASS  {name}: {actual}")

methods = [surface_area_simulate, surface_area_four_dirs, surface_area_up_left,
           Solution().surfaceArea, SolutionFourDirs().surfaceArea]

cases = [
    ("đề VD1", [[1, 2], [3, 4]], 34),
    ("đề VD2 (giếng)", [[1, 1, 1], [1, 0, 1], [1, 1, 1]], 32),
    ("đề VD3", [[2, 2, 2], [2, 1, 2], [2, 2, 2]], 46),
    ("1 khối", [[1]], 6),
    ("ô 0 duy nhất", [[0]], 0),
    ("tháp cao 50", [[50]], 202),
    ("hai tháp chéo không chạm", [[1, 0], [0, 1]], 12),
    ("hai tháp kề", [[2, 1], [0, 0]], 4 * 2 + 2 + 4 * 1 + 2 - 2 * 1),
    ("toàn 0", [[0, 0], [0, 0]], 0),
]
for name, g, exp in cases:
    for f in methods:
        assert f(g) == exp, (name, f)
    check(name, surface_area_up_left(g), exp)

PASS  đề VD1: 34
PASS  đề VD2 (giếng): 32
PASS  đề VD3: 46
PASS  1 khối: 6
PASS  ô 0 duy nhất: 0
PASS  tháp cao 50: 202
PASS  hai tháp chéo không chạm: 12
PASS  hai tháp kề: 14
PASS  toàn 0: 0


In [15]:
import random
random.seed(892)
for _ in range(300):
    n = random.randint(1, 5)
    g = [[random.choice([0, 0, 1, 2, 3, 5]) for _ in range(n)] for _ in range(n)]
    expected = surface_area_simulate(g)
    for f in methods:
        assert f(g) == expected
print("300 lưới ngẫu nhiên: 5 cách luôn khớp mô phỏng từng khối.")
big = [[50] * 50 for _ in range(50)]
print("Lưới 50x50 toàn 50:", Solution().surfaceArea(big), "= 2·2500 (đỉnh+đáy) + 4·50·50 (4 mặt ngoài) =", 2 * 2500 + 4 * 50 * 50)

300 lưới ngẫu nhiên: 5 cách luôn khớp mô phỏng từng khối.
Lưới 50x50 toàn 50: 15000 = 2·2500 (đỉnh+đáy) + 4·50·50 (4 mặt ngoài) = 15000


## 12. Lỗi người mới hay gặp

### Lỗi 1: Cộng `4v + 2` cho cả ô `v = 0`
Ô trống bị cộng 2 (đỉnh+đáy ma) → sai.

In [16]:
def wrong_zero_cell(grid):
    n = len(grid); area = 0
    for i in range(n):
        for j in range(n):
            v = grid[i][j]
            area += 4 * v + 2                     # SAI: không kiểm v > 0
            if i > 0: area -= 2 * min(v, grid[i - 1][j])
            if j > 0: area -= 2 * min(v, grid[i][j - 1])
    return area

g = [[1, 1, 1], [1, 0, 1], [1, 1, 1]]
print("Sai:", wrong_zero_cell(g), "| Đúng:", Solution().surfaceArea(g))

Sai: 34 | Đúng: 32


### Lỗi 2: Xét đủ 4 hướng nhưng vẫn trừ `2·min` → trừ **hai lần** mỗi cặp
Nếu dùng cả 4 hàng xóm thì chỉ trừ `min` (một lần mỗi phía), hoặc chỉ xét trên/trái và trừ `2·min`.

In [17]:
def wrong_double_subtract(grid):
    n = len(grid); area = 0
    for i in range(n):
        for j in range(n):
            v = grid[i][j]
            if v: area += 4 * v + 2
            for ni, nj in ((i - 1, j), (i + 1, j), (i, j - 1), (i, j + 1)):
                if 0 <= ni < n and 0 <= nj < n:
                    area -= 2 * min(v, grid[ni][nj])   # SAI: mỗi cặp bị trừ 2 lần
    return area

print("Sai:", wrong_double_subtract([[1, 2], [3, 4]]), "| Đúng:", Solution().surfaceArea([[1, 2], [3, 4]]))

Sai: 20 | Đúng: 34


### Lỗi 3: Không kiểm biên, dùng `grid[i-1][j]` khi `i = 0`
`grid[-1][j]` lấy **hàng cuối** — không báo lỗi mà tính sai (coi hàng đầu và hàng cuối là kề nhau).

In [18]:
def wrong_no_bounds(grid):
    n = len(grid); area = 0
    for i in range(n):
        for j in range(n):
            v = grid[i][j]
            if v: area += 4 * v + 2
            area -= 2 * min(v, grid[i - 1][j])          # SAI khi i = 0
            area -= 2 * min(v, grid[i][j - 1])          # SAI khi j = 0
    return area

print("Sai:", wrong_no_bounds([[1, 2], [3, 4]]), "| Đúng:", Solution().surfaceArea([[1, 2], [3, 4]]))

Sai: 20 | Đúng: 34


### Lỗi 4: Nhầm với bài 883 (bóng chiếu)
Bài 883 dùng **max** theo hàng/cột và không tính mặt "lõm vào". Bài này là **bề mặt thật**: mặt bên trong giếng, bậc thang giữa các tháp đều được tính.

### Lỗi 5: Quên mặt đáy
Đề nói rõ đáy được tính → `+2` (đỉnh và đáy), không phải `+1`.

## 13. Mẹo nhận dạng pattern

- "Diện tích bề mặt / chu vi của các ô/khối **dán nhau**" → **tổng riêng lẻ − phần tiếp xúc** (mỗi tiếp xúc trừ 2), hoặc **đếm cạnh/mặt lộ ra** theo 4 hướng.
- Muốn mỗi cặp kề đếm đúng 1 lần → chỉ xét hàng xóm **trên + trái** (hoặc dưới + phải).
- Lưới + hàng xóm → luôn viết **kiểm tra biên** (đừng tin chỉ số âm của Python).
- Nghi ngờ công thức → viết **mô phỏng nhỏ** làm oracle rồi test ngẫu nhiên.

Bài cùng họ: 463 Island Perimeter (bản 2D: `4·số ô − 2·số cặp kề`), 883 Projection Area, 695 Max Area of Island.

## 14. Biến thể

### 14.1. Bản 2D — LeetCode 463 Island Perimeter
Mỗi ô đất 4 cạnh, mỗi cặp ô đất kề nhau trừ 2. Đúng cấu trúc cách 3!

In [19]:
def island_perimeter(grid: list[list[int]]) -> int:
    rows, cols = len(grid), len(grid[0])
    p = 0
    for i in range(rows):
        for j in range(cols):
            if grid[i][j] == 1:
                p += 4
                if i > 0 and grid[i - 1][j] == 1:
                    p -= 2
                if j > 0 and grid[i][j - 1] == 1:
                    p -= 2
    return p

print(island_perimeter([[0, 1, 0, 0], [1, 1, 1, 0], [0, 1, 0, 0], [1, 1, 0, 0]]))   # 16

16


### 14.2. Không tính mặt đáy (khối đặt trên sàn)
Chỉ cần đổi `+2` thành `+1` cho mỗi tháp khác 0.

### 14.3. Lưới chữ nhật `m × n`
Thay `n` bằng số hàng và số cột riêng — cách 3 không cần sửa gì khác.

In [20]:
def surface_area_no_bottom(grid):
    rows, cols = len(grid), len(grid[0])
    area = 0
    for i in range(rows):
        for j in range(cols):
            v = grid[i][j]
            if v:
                area += 4 * v + 1
            if i > 0:
                area -= 2 * min(v, grid[i - 1][j])
            if j > 0:
                area -= 2 * min(v, grid[i][j - 1])
    return area

print("Không đáy [[1,2],[3,4]]:", surface_area_no_bottom([[1, 2], [3, 4]]), "= 34 - 4")
print("Lưới 1x3 [[1,2,1]]:", surface_area_no_bottom([[1, 2, 1]]) + 3, "(cộng lại 3 đáy để có bản có đáy)")

Không đáy [[1,2],[3,4]]: 30 = 34 - 4
Lưới 1x3 [[1,2,1]]: 18 (cộng lại 3 đáy để có bản có đáy)


## 15. Bài tập tự luyện

Hàm TODO chỉ định nghĩa, không được gọi khi chạy notebook.

### Bài 1 — Tính tay
Tính diện tích bề mặt của `[[3, 0], [0, 3]]` và của `[[3, 3], [0, 0]]`. Giải thích vì sao khác nhau.

### Bài 2 — Dự đoán output
Với `grid = [[1, 2], [3, 4]]`, các biểu thức sau cho gì: `grid[-1][0]`, `max(2 - 5, 0)`, `min(4, 0)`?

### Bài 3 — Một tháp đơn

In [21]:
def tower_area(v: int) -> int:
    # TODO: 0 nếu v == 0, ngược lại 4v + 2
    pass

# assert tower_area(0) == 0 and tower_area(1) == 6 and tower_area(3) == 14

### Bài 4 — Tự viết lại cách 3 (trên/trái)

In [22]:
def surface_area_exercise(grid) -> int:
    # TODO
    pass

# assert surface_area_exercise([[1, 2], [3, 4]]) == 34
# assert surface_area_exercise([[1, 1, 1], [1, 0, 1], [1, 1, 1]]) == 32

### Bài 5 — Biến thể: Island Perimeter cho lưới chữ nhật, viết theo kiểu "đếm cạnh lộ ra" (4 hướng)

In [23]:
def island_perimeter_four_dirs(grid) -> int:
    # TODO: mỗi ô đất, mỗi hướng: ra ngoài lưới hoặc gặp nước -> +1
    pass

# assert island_perimeter_four_dirs([[0,1,0,0],[1,1,1,0],[0,1,0,0],[1,1,0,0]]) == 16
# assert island_perimeter_four_dirs([[1]]) == 4

### Bài 6 — Mini challenge: diện tích **bị che** (tổng mặt dán)
Trả về số mặt bị che do dán nhau (cả theo chiều đứng lẫn chiều ngang), tức `6 · (tổng số khối) − surfaceArea`.
Viết trực tiếp không gọi `surfaceArea`: tính tổng `2(v−1)` cho mỗi tháp `v>0` và `2·min` cho mỗi cặp kề.

In [24]:
def hidden_faces(grid) -> int:
    # TODO
    pass

# assert hidden_faces([[1, 2], [3, 4]]) == 6 * 10 - 34

## 16. Gợi ý 3 mức

| Bài | Mức 1 | Mức 2 | Mức 3 |
|---|---|---|---|
| 1 | hai tháp có chạm không? | chéo nhau thì không kề | `14+14=28` và `28 − 2·3 = 22` |
| 2 | chỉ số âm | max với 0 | `3`, `0`, `0` |
| 3 | tháp riêng | `if v == 0` | `return 4 * v + 2` |
| 4 | trên + trái | `if i > 0`, `if j > 0` | trừ `2 * min(v, nb)` |
| 5 | 4 hướng | kiểm biên | ra ngoài hoặc `grid[ni][nj] == 0` → `+1` |
| 6 | 6·khối − bề mặt | dán đứng `2(v−1)` | cặp ngang/dọc `2·min` |

## 17. Đáp án tham khảo

**Bài 1:** `[[3,0],[0,3]]`: hai tháp chéo nhau, **không kề** → `14 + 14 = 28`. `[[3,3],[0,0]]`: kề nhau, tầng chung 3 → `28 − 2·3 = 22`.

**Bài 2:** `grid[-1][0] = 3` (hàng cuối), `max(-3, 0) = 0`, `min(4, 0) = 0`.

In [25]:
def ans_tower_area(v):
    return 4 * v + 2 if v > 0 else 0

def ans_surface_area(grid):
    n = len(grid); area = 0
    for i in range(n):
        for j in range(n):
            v = grid[i][j]
            area += ans_tower_area(v)
            if i > 0: area -= 2 * min(v, grid[i - 1][j])
            if j > 0: area -= 2 * min(v, grid[i][j - 1])
    return area

def ans_island_perimeter_four_dirs(grid):
    rows, cols = len(grid), len(grid[0]); p = 0
    for i in range(rows):
        for j in range(cols):
            if grid[i][j] != 1:
                continue
            for ni, nj in ((i - 1, j), (i + 1, j), (i, j - 1), (i, j + 1)):
                if not (0 <= ni < rows and 0 <= nj < cols) or grid[ni][nj] == 0:
                    p += 1
    return p

def ans_hidden_faces(grid):
    n = len(grid); hidden = 0
    for i in range(n):
        for j in range(n):
            v = grid[i][j]
            if v > 0:
                hidden += 2 * (v - 1)
            if i > 0: hidden += 2 * min(v, grid[i - 1][j])
            if j > 0: hidden += 2 * min(v, grid[i][j - 1])
    return hidden

assert Solution().surfaceArea([[3, 0], [0, 3]]) == 28 and Solution().surfaceArea([[3, 3], [0, 0]]) == 22
g = [[1, 2], [3, 4]]
assert g[-1][0] == 3 and max(2 - 5, 0) == 0 and min(4, 0) == 0
assert ans_tower_area(0) == 0 and ans_tower_area(1) == 6 and ans_tower_area(3) == 14
assert ans_surface_area([[1, 2], [3, 4]]) == 34
assert ans_surface_area([[1, 1, 1], [1, 0, 1], [1, 1, 1]]) == 32
assert ans_island_perimeter_four_dirs([[0,1,0,0],[1,1,1,0],[0,1,0,0],[1,1,0,0]]) == 16
assert ans_island_perimeter_four_dirs([[1]]) == 4
assert ans_hidden_faces([[1, 2], [3, 4]]) == 6 * 10 - 34
random.seed(1)
for _ in range(100):
    n = random.randint(1, 4)
    g = [[random.randint(0, 4) for _ in range(n)] for _ in range(n)]
    assert ans_hidden_faces(g) == 6 * sum(map(sum, g)) - Solution().surfaceArea(g)
print("Đáp án tham khảo PASS toàn bộ test!")

Đáp án tham khảo PASS toàn bộ test!


## 18. Template nhớ nhanh và checklist submit

```python
class Solution:
    def surfaceArea(self, grid):
        n, area = len(grid), 0
        for i in range(n):
            for j in range(n):
                v = grid[i][j]
                if v:
                    area += 4 * v + 2                       # tháp đứng riêng
                if i > 0:
                    area -= 2 * min(v, grid[i - 1][j])      # dán với ô trên
                if j > 0:
                    area -= 2 * min(v, grid[i][j - 1])      # dán với ô trái
        return area
```

**Câu nhớ:** *"Mỗi tháp 4v + 2; mỗi cặp kề mất 2·min."*

**Checklist:**
- [ ] Ô `v = 0` không cộng `+2`?
- [ ] Mỗi cặp kề trừ **đúng một lần** (chỉ trên/trái), trừ `2·min`?
- [ ] Kiểm biên `i > 0`, `j > 0` (không để `grid[-1]` lén chạy)?
- [ ] Có tính mặt đáy (`+2`, không phải `+1`)?
- [ ] Test ví dụ "giếng" có ô 0 ở giữa?